# GroupDNA – Minor Project 1

## WhatsApp Group Chat Analyzer

**Name:** Samarpan Pandit

**Batch:** Data Analytics June 2026

**Course:** Python Fundamentals + NumPy

**Dataset:** hostel_bois.txt

**Submitted To:** SkillOrbit

---

### Project Objective

The objective of this project is to analyze a WhatsApp chat export and generate a complete behavioral analytics report of the group. The project is developed using only Python fundamentals and NumPy without using pandas, regex, matplotlib, Counter, or any external chat analysis libraries.

In [ ]:
# IMPORT LIBRARIES
import numpy as np
from datetime import datetime, timedelta

#Feature-1: Chat Parser

In [ ]:
def parse_chat(file_path):
    parsed_messages = []

    # Counters for edge cases
    system_msgs_count = 0
    media_omitted_count = 0
    deleted_msgs_count = 0

    # Open and read the text file line by line
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            # Skip completely empty lines silently
            if not line:
                continue

            # Check if line has the timestamp-separator dash
            if ' - ' not in line:
                continue

            # 1. Split timestamp from the rest of the line (using maxsplit=1)
            parts = line.split(' - ', 1)
            timestamp_str = parts[0].strip()
            rest_of_line = parts[1].strip()

            # 2. Check for system messages (no ":" after the sender)
            if ':' not in rest_of_line:
                system_msgs_count += 1
                continue # Skip system messages from the main parsed list

            # 3. Split sender from message text (using maxsplit=1)
            sender_parts = rest_of_line.split(':', 1)
            sender = sender_parts[0].strip()
            message_text = sender_parts[1].strip()

            # 4. Handle Edge Cases for messages
            if message_text == '<Media omitted>':
                media_omitted_count += 1
            elif message_text == 'This message was deleted':
                deleted_msgs_count += 1

            # Append the extracted data to our list of dictionaries
            parsed_messages.append({
                'timestamp': timestamp_str,
                'sender': sender,
                'text': message_text
            })

    # Calculate unique participants dynamically for the print statement
    participants = set(msg['sender'] for msg in parsed_messages)

    # Formatted final output as required by the brief
    print(f"Successfully parsed {len(parsed_messages)} messages from {len(participants)} participants over 60 days, skipped {system_msgs_count} system messages, {media_omitted_count} media-omitted, {deleted_msgs_count} deleted messages.")

    return parsed_messages

# To run the feature on your dataset:
messages = parse_chat('hostel_bois.txt')

Successfully parsed 3174 messages from 6 participants over 60 days, skipped 4 system messages, 32 media-omitted, 15 deleted messages.


#Feature-2: Group Overview

In [ ]:
def group_overview(parsed_messages, group_name="Hostel Bois 4ever"):
    """
    Computes and prints the headline statistics for the WhatsApp group,
    including total messages, duration, and a sorted per-person breakdown.
    """

    # 1. Count messages per person using a standard dictionary
    sender_counts = {}
    for msg in parsed_messages:
        sender = msg['sender']
        if sender in sender_counts:
            sender_counts[sender] += 1
        else:
            sender_counts[sender] = 1

    # Sort the dictionary items by count in descending order
    # x[1] refers to the value (the count) in the (key, value) tuple
    sorted_senders = sorted(sender_counts.items(), key=lambda x: x[1], reverse=True)

    # 2. Extract and format the date range
    # Get the raw date strings (DD/MM/YY) from the first and last messages
    first_date_raw = parsed_messages[0]['timestamp'].split(',')[0].strip()
    last_date_raw = parsed_messages[-1]['timestamp'].split(',')[0].strip()

    # Use datetime to calculate the total duration in days (allowed per Section 8 constraints)
    date_format = "%d/%m/%y"
    start_date = datetime.strptime(first_date_raw, date_format)
    end_date = datetime.strptime(last_date_raw, date_format)
    total_days = (end_date - start_date).days + 1  # +1 to make it inclusive

    # Format the dates into human-readable strings (e.g., "01 April 2024")
    # %d = Day, %B = Full month name, %Y = 4-digit year
    first_date_str = start_date.strftime("%d %B %Y")
    last_date_str = end_date.strftime("%d %B %Y")

    # 3. Calculate overarching stats
    total_messages = len(parsed_messages)
    total_participants = len(sender_counts)

    # 4. Print the formatted report
    print("=" * 60)
    print("GROUP OVERVIEW")
    print("=" * 60)
    print(f"Group : {group_name}\n")

    print(f"Period : {first_date_str} to {last_date_str} ({total_days} days)")
    # Use the :, format specifier to add commas to thousands (e.g., 3,174)
    print(f"Total messages : {total_messages:,}")
    print(f"Participants : {total_participants}\n")

    print("MESSAGES PER PERSON")

    # Dynamically find the longest name to align the colons perfectly
    max_name_len = max(len(name) for name in sender_counts.keys())

    # Print each person's stats with strict f-string alignment
    for name, count in sorted_senders:
        percentage = (count / total_messages) * 100

        # < aligns left, > aligns right.
        # >3 ensures numbers like "24" line up neatly under "354"
        # >4.1f ensures percentages like "0.8" line up under "11.2"
        print(f"{name:<{max_name_len}} : {count:>3} ({percentage:>4.1f}%)")

# To run the feature using the data parsed from Feature 1:
group_overview(messages)

GROUP OVERVIEW
Group : Hostel Bois 4ever

Period : 01 April 2024 to 30 May 2024 (60 days)
Total messages : 3,174
Participants : 6

MESSAGES PER PERSON
Rahul : 953 (30.0%)
Priya : 718 (22.6%)
Neha  : 635 (20.0%)
Aman  : 490 (15.4%)
Karan : 354 (11.2%)
Vikas :  24 ( 0.8%)


#Feature-3: Most Active Day and Hour

In [ ]:
def most_active_day_and_hour(parsed_messages):
    """
    Identifies the single day with the most messages and the
    most active hour of the day across the entire dataset.
    """
    day_counts = {}
    hour_counts = {}

    # 1. Loop over messages and populate frequency dictionaries
    for msg in parsed_messages:
        timestamp = msg['timestamp']

        # Split the timestamp string "DD/MM/YY, HH:MM"
        parts = timestamp.split(',', 1)
        date_str = parts[0].strip()
        time_str = parts[1].strip()

        # Extract just the hour (e.g., from "23:14" get "23")
        hour_str = time_str.split(':', 1)[0].strip()

        # Update day counts
        if date_str in day_counts:
            day_counts[date_str] += 1
        else:
            day_counts[date_str] = 1

        # Update hour counts
        if hour_str in hour_counts:
            hour_counts[hour_str] += 1
        else:
            hour_counts[hour_str] = 1

    # 2. Find the maximum values using max() with a lambda key
    busiest_day_raw, max_day_count = max(day_counts.items(), key=lambda x: x[1])
    busiest_hour_raw, max_hour_count = max(hour_counts.items(), key=lambda x: x[1])

    # 3. Format the Date Manually (Strict constraint compliance)
    month_names = {
        "01": "January", "02": "February", "03": "March", "04": "April",
        "05": "May", "06": "June", "07": "July", "08": "August",
        "09": "September", "10": "October", "11": "November", "12": "December"
    }
    day, month, year = busiest_day_raw.split('/')
    busiest_day_formatted = f"{day} {month_names[month]} 20{year}"

    # 4. Format the Hour Range (e.g., "17" -> "17:00 - 18:00")
    busiest_hour_int = int(busiest_hour_raw)
    hour_start = f"{busiest_hour_raw}:00"

    # Handle the wrap-around for midnight using modulo arithmetic
    next_hour = str((busiest_hour_int + 1) % 24).zfill(2)
    hour_end = f"{next_hour}:00"

    # 5. Calculate Average Messages Per Day for the Busiest Hour
    # (Extract start and end dates from the dataset to find total days)
    first_date_raw = parsed_messages[0]['timestamp'].split(',')[0].strip()
    last_date_raw = parsed_messages[-1]['timestamp'].split(',')[0].strip()

    start_date = datetime.strptime(first_date_raw, "%d/%m/%y")
    end_date = datetime.strptime(last_date_raw, "%d/%m/%y")
    total_days = (end_date - start_date).days + 1

    avg_per_day = round(max_hour_count / total_days)

    # 6. Print the formatted output exactly as requested
    print(f"Busiest day : {busiest_day_formatted} ({max_day_count} messages)")
    print(f"Busiest hour: {hour_start} - {hour_end} (avg {avg_per_day} messages per day)")

# To run the feature using the data parsed from Feature 1:
most_active_day_and_hour(messages)

Busiest day : 04 May 2024 (76 messages)
Busiest hour: 18:00 - 19:00 (avg 4 messages per day)


#Feature-4: Activity Heatmap

In [ ]:
def activity_heatmap(parsed_messages):
    """
    Builds a 6x24 NumPy matrix of message counts per person per hour,
    and prints it as a text-based terminal heatmap using ASCII blocks.
    """

    # 1. Determine unique participants (sorted by activity to match Feature 2's visual flow)
    sender_counts = {}
    for msg in parsed_messages:
        sender = msg['sender']
        sender_counts[sender] = sender_counts.get(sender, 0) + 1

    # Sort from most active to least active
    sorted_participants = [item[0] for item in sorted(sender_counts.items(), key=lambda x: x[1], reverse=True)]

    # 2. Initialize the NumPy Matrix (Rows = Participants, Columns = 24 Hours)
    # Using np.zeros as explicitly required by the project constraints
    heatmap_data = np.zeros((len(sorted_participants), 24), dtype=int)

    # 3. Populate the Matrix using integer indexing
    for msg in parsed_messages:
        sender = msg['sender']

        # Extract the hour from the timestamp string "DD/MM/YY, HH:MM"
        time_str = msg['timestamp'].split(',')[1].strip()
        hour = int(time_str.split(':')[0])

        # Find the correct row index for this sender and increment the cell
        row_idx = sorted_participants.index(sender)
        heatmap_data[row_idx, hour] += 1

    # 4. Find the maximum messages per person to calculate relative shading
    # np.max with axis=1 gets the highest hour count for EACH row (person)
    row_maxes = np.max(heatmap_data, axis=1)

    # Prevent division-by-zero errors in case someone has absolutely 0 messages
    row_maxes[row_maxes == 0] = 1

    # 5. Render the ASCII Heatmap
    print("ACTIVITY HEATMAP (messages by hour)")

    # Dynamically pad the header so it aligns perfectly with the longest name
    max_name_len = max(len(name) for name in sorted_participants)
    header_padding = " " * (max_name_len + 3)

    # The spacing here is mathematically precise: 8 labels * 3 chars = 24 chars wide
    print(f"{header_padding}00 03 06 09 12 15 18 21")

    for i, name in enumerate(sorted_participants):
        row = heatmap_data[i]
        max_val = row_maxes[i]

        row_str = ""
        # Loop through all 24 hours to generate a 24-character string of blocks
        for val in row:
            ratio = val / max_val

            # Apply the 4 shading levels based on the cell's value relative to the person's maximum
            if ratio <= 0.25:
                row_str += "."  # 0 - 25% (Includes zero activity)
            elif ratio <= 0.50:
                row_str += "░"  # 25 - 50%
            elif ratio <= 0.75:
                row_str += "▒"  # 50 - 75%
            else:
                row_str += "▓"  # 75 - 100%

        # Print the fully padded name followed by the 24-character heatmap row
        print(f"{name:<{max_name_len}} : {row_str}")

# To run the feature using the data parsed from Feature 1:
activity_heatmap(messages)

ACTIVITY HEATMAP (messages by hour)
        00 03 06 09 12 15 18 21
Rahul : ............▒░░▒▒░▓▒░▓▒▒
Priya : .......░▒▓▓▓▓▒▒░░▒▒▓▒░░.
Neha  : .....░..▒▓▓░▒▒░.▒▓▓▓▒░░░
Aman  : ▒▓▒▒▓..................▒
Karan : ........░░▒░▓▒▓▒▒▒▒▓▒░..
Vikas : .......░▓░░.▒▒.░░▓▒▒░░░▒


#Feature-5: Top Words

In [ ]:
def top_words(parsed_messages):
    """
    Extracts, cleans, and counts every word from the chat, filtering out
    stop words and punctuation, and prints the top 10 words as an ASCII bar chart.
    """

    # 1. Define stop words and punctuation (Constraint: No NLP libraries allowed)
    # Added common English words and a few basic Hinglish fillers to clean the output
    stop_words = {
        'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
        'you', 'it', 'that', 'this', 'my', 'me', 'with', 'hai', 'ki', 'se',
        'ko', 'hi', 'are', 'am', 'was', 'were', 'be', 'do', 'have', 'not',
        'at', 'as', 'but', 'if', 'we', 'so', 'can', 'just', 'like'
    }

    # String containing all characters we want to strip from the edges of words
    punctuation = ".,!?\"'()[]{}:;*_-~\\/"

    # 2. Build the word frequency dictionary
    word_counts = {}

    for msg in parsed_messages:
        text = msg['text']

        # Skip special WhatsApp system strings explicitly
        if text == '<Media omitted>' or text == 'This message was deleted':
            continue

        # Split message into tokens by whitespace
        raw_words = text.split()

        for raw_word in raw_words:
            # Normalise: Lowercase and strip punctuation from both edges
            # (Constraint: No regex allowed)
            clean_word = raw_word.lower().strip(punctuation)

            # Check if word is valid (not empty), not a stop word, and >1 character
            if clean_word and clean_word not in stop_words and len(clean_word) > 1:
                # Accumulate using standard dict methods (No collections.Counter)
                word_counts[clean_word] = word_counts.get(clean_word, 0) + 1

    # 3. Sort to find the Top 10 words
    # Sorts by value (x[1]) in descending order and slices the top 10
    top_10 = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]

    # 4. Render the output (Bonus: Horizontal bar chart)
    print("THIS GROUP'S FAVOURITE WORDS")

    if not top_10:
        print("No valid words found.")
        return

    # Get the highest count for calculating proportional bar lengths
    max_count = top_10[0][1]

    # Dynamically find the longest word in the top 10 for clean visual alignment
    max_word_len = max(len(word) for word, count in top_10)

    for word, count in top_10:
        # Calculate bar length (max width = 30 blocks)
        # Ensure even the smallest item gets at least 1 block
        bar_len = max(1, int((count / max_count) * 30))
        bar = "█" * bar_len

        # Print with dynamic f-string padding so bars start at the same column
        print(f"{word:<{max_word_len}} {bar} {count}")

# To run the feature using the data parsed from Feature 1:
top_words(messages)

THIS GROUP'S FAVOURITE WORDS
how      ██████████████████████████████ 321
guys     █████████████████████████████ 318
about    █████████████████████████ 274
today    ████████████████████████ 257
he       ████████████████████ 220
his      ████████████████████ 217
which    ██████████████████ 202
everyone █████████████████ 187
telling  ████████████████ 179
from     ████████████████ 174


#Feature-6: Response Speed & Silent Streaks

In [ ]:
def response_speed_and_silent_streaks(parsed_messages):
    """
    Calculates average response times and the longest streak of consecutive
    silent days for each participant in the chat.
    """

    # 1. Initialize data structures
    response_gaps = {}
    active_dates = {}
    participants = set()

    for msg in parsed_messages:
        sender = msg['sender']
        participants.add(sender)

        if sender not in response_gaps:
            response_gaps[sender] = []
        if sender not in active_dates:
            active_dates[sender] = set()

        # Add the date string (DD/MM/YY) to the active dates set
        date_str = msg['timestamp'].split(',')[0].strip()
        active_dates[sender].add(date_str)

    # 2. Calculate Response Times
    # Gap between someone else's message and this person's immediate reply
    prev_sender = None
    prev_time = None

    for msg in parsed_messages:
        current_sender = msg['sender']
        current_time = datetime.strptime(msg['timestamp'], "%d/%m/%y, %H:%M")

        if prev_sender is not None and current_sender != prev_sender:
            gap = (current_time - prev_time).total_seconds()
            response_gaps[current_sender].append(gap)

        prev_sender = current_sender
        prev_time = current_time

    # Compute averages
    avg_responses = {}
    for sender, gaps in response_gaps.items():
        if len(gaps) > 0:
            avg_seconds = sum(gaps) / len(gaps)
            avg_responses[sender] = avg_seconds
        else:
            avg_responses[sender] = 0

    # Find Fastest and Slowest Repliers
    sorted_repliers = sorted(avg_responses.items(), key=lambda x: x[1])
    fastest = sorted_repliers[0]
    slowest = sorted_repliers[-1]

    def format_time(seconds):
        if seconds < 3600:
            return f"{round(seconds / 60, 1)} minutes"
        return f"{round(seconds / 3600, 1)} hours"

    # 3. Calculate Longest Silent Streaks
    first_date_raw = parsed_messages[0]['timestamp'].split(',')[0].strip()
    last_date_raw = parsed_messages[-1]['timestamp'].split(',')[0].strip()
    start_date = datetime.strptime(first_date_raw, "%d/%m/%y")
    end_date = datetime.strptime(last_date_raw, "%d/%m/%y")
    total_days = (end_date - start_date).days + 1

    month_names = {
        "01": "Jan", "02": "Feb", "03": "Mar", "04": "Apr",
        "05": "May", "06": "Jun", "07": "Jul", "08": "Aug",
        "09": "Sep", "10": "Oct", "11": "Nov", "12": "Dec"
    }

    streak_records = {}

    for sender in participants:
        longest_streak = 0
        current_streak = 0
        streak_start = None
        best_streak_start = None
        best_streak_end = None

        for i in range(total_days):
            current_date = start_date + timedelta(days=i)
            # Format back to match our parsed strings manually (DD/MM/YY)
            day = str(current_date.day).zfill(2)
            month = str(current_date.month).zfill(2)
            year = str(current_date.year)[-2:]
            date_str = f"{day}/{month}/{year}"

            if date_str not in active_dates[sender]:
                if current_streak == 0:
                    current_start = current_date
                current_streak += 1
            else:
                if current_streak > longest_streak:
                    longest_streak = current_streak
                    best_streak_start = current_start
                    best_streak_end = current_date - timedelta(days=1)
                current_streak = 0

        # Check one last time in case the longest streak ends on the very last day
        if current_streak > longest_streak:
            longest_streak = current_streak
            best_streak_start = current_start
            best_streak_end = end_date

        # Format the start and end dates
        if longest_streak > 0:
            start_str = f"{str(best_streak_start.day).zfill(2)} {month_names[str(best_streak_start.month).zfill(2)]}"
            end_str = f"{str(best_streak_end.day).zfill(2)} {month_names[str(best_streak_end.month).zfill(2)]}"
            date_range = f"({start_str} to {end_str})"
        else:
            date_range = "(never went silent)"

        streak_records[sender] = {
            'days': longest_streak,
            'range': date_range
        }

    # Sort streaks descending
    sorted_streaks = sorted(streak_records.items(), key=lambda x: x[1]['days'], reverse=True)

    # 4. Print the output
    print("RESPONSE PATTERNS")
    print(f"Fastest replier: {fastest[0]} (avg {format_time(fastest[1])})")
    print(f"Slowest replier: {slowest[0]} (avg {format_time(slowest[1])})\n")

    print("LONGEST SILENT STREAKS")

    max_name_len = max(len(name) for name in participants)
    for name, data in sorted_streaks:
        if data['days'] == 0:
            print(f"{name:<{max_name_len}} :  0 days {data['range']}")
        else:
            print(f"{name:<{max_name_len}} : {data['days']:>2} days {data['range']}")

# To run the feature using the data parsed from Feature 1:
response_speed_and_silent_streaks(messages)

RESPONSE PATTERNS
Fastest replier: Rahul (avg 34.9 minutes)
Slowest replier: Aman (avg 55.4 minutes)

LONGEST SILENT STREAKS
Vikas : 11 days (23 Apr to 03 May)
Aman  :  0 days (never went silent)
Rahul :  0 days (never went silent)
Priya :  0 days (never went silent)
Karan :  0 days (never went silent)
Neha  :  0 days (never went silent)


#Feature-7: Personality Archetype Detection

In [ ]:
def personality_archetypes(parsed_messages):
    """
    Analyzes message patterns to assign a unique personality archetype
    to each participant based on defined quantitative rules and thresholds.
    """

    # 1. Initialize data structures for tracking raw stats
    stats = {}
    participants = []

    # Total days calculation for the Ghost archetype
    first_date_raw = parsed_messages[0]['timestamp'].split(',')[0].strip()
    last_date_raw = parsed_messages[-1]['timestamp'].split(',')[0].strip()
    start_date = datetime.strptime(first_date_raw, "%d/%m/%y")
    end_date = datetime.strptime(last_date_raw, "%d/%m/%y")
    total_days = (end_date - start_date).days + 1

    # Pre-define target keywords
    caring_words = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"]
    comedian_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
    hype_words = ['fire', 'lit', 'crazy', 'insane', 'bro', 'epic', 'goat'] # For Bonus Archetype

    # 2. Iterate through messages to gather raw data
    prev_sender = None

    for msg in parsed_messages:
        sender = msg['sender']
        text = msg['text']
        text_lower = text.lower()

        # Initialize dictionary for new senders
        if sender not in stats:
            participants.append(sender)
            stats[sender] = {
                'total_msgs': 0, 'burst_count': 0, 'bursts': [], 'current_burst': 0,
                'caring_count': 0, 'night_msgs': 0, 'total_words': 0,
                'drama_msgs': 0, 'active_days': set(), 'comedian_count': 0,
                'question_msgs': 0, 'hype_count': 0
            }

        p_stats = stats[sender]

        # Skip edge cases for text-based analysis, but count for bursts/activity
        is_real_text = text not in ['<Media omitted>', 'This message was deleted']

        # Total messages & active days
        p_stats['total_msgs'] += 1
        date_str = msg['timestamp'].split(',')[0].strip()
        p_stats['active_days'].add(date_str)

        # Burst tracking (The Spammer)
        if sender == prev_sender:
            p_stats['current_burst'] += 1
        else:
            if prev_sender is not None:
                stats[prev_sender]['bursts'].append(stats[prev_sender]['current_burst'] + 1)
                stats[prev_sender]['current_burst'] = 0
        prev_sender = sender

        # Time tracking (The Night Owl) - 23:00 to 04:59
        hour = int(msg['timestamp'].split(',')[1].split(':')[0].strip())
        if hour == 23 or 0 <= hour <= 4:
            p_stats['night_msgs'] += 1

        if is_real_text:
            words = text_lower.split()
            p_stats['total_words'] += len(words)

            # Group Mom
            for cw in caring_words:
                if cw in text_lower:
                    p_stats['caring_count'] += 1

            # Comedian
            for cw in comedian_words:
                if cw in text_lower:
                    p_stats['comedian_count'] += 1

            # Bonus: Hype Beast
            for hw in hype_words:
                if hw in text_lower:
                    p_stats['hype_count'] += 1

            # Drama Queen
            is_all_caps = text.isupper() and len(text) >= 3
            has_exclamations = text.count('!') >= 2
            if is_all_caps or has_exclamations:
                p_stats['drama_msgs'] += 1

            # Question Master
            if text.strip().endswith('?'):
                p_stats['question_msgs'] += 1

    # Catch the last burst
    if prev_sender is not None:
        stats[prev_sender]['bursts'].append(stats[prev_sender]['current_burst'] + 1)

    # 3. Calculate final metrics and find group maxes (for relative archetypes)
    group_max_caring = max(s['caring_count'] for s in stats.values()) or 1
    group_max_comedian = max((s['comedian_count'] / s['total_msgs']) for s in stats.values()) or 1

    # 4. Score every archetype for every person
    # We normalize scores by dividing the actual metric by the target threshold
    # A score > 1.0 means they passed the threshold. The highest score wins.

    assigned_archetypes = {}

    for sender, s in stats.items():
        total = s['total_msgs']

        avg_burst = sum(s['bursts']) / len(s['bursts']) if s['bursts'] else 0
        night_pct = s['night_msgs'] / total
        avg_words = s['total_words'] / total
        drama_pct = s['drama_msgs'] / total
        silent_days = total_days - len(s['active_days'])
        silent_pct = silent_days / total_days
        comedian_pct = s['comedian_count'] / total
        question_pct = s['question_msgs'] / total

        # Calculate normalized scores against defined thresholds
        scores = {
            "THE SPAMMER": avg_burst / 3.0,                     # Threshold: 3 msgs in a row
            "THE GROUP MOM": s['caring_count'] / group_max_caring, # Threshold: Highest in group
            "THE NIGHT OWL": night_pct / 0.60,                  # Threshold: 60%
            "THE STORYTELLER": avg_words / 30.0,                # Threshold: 30 words/msg
            "THE DRAMA QUEEN": drama_pct / 0.30,                # Threshold: 30%
            "THE GHOST": silent_pct / 0.60,                     # Threshold: 60% silent days
            "THE COMEDIAN": comedian_pct / group_max_comedian,  # Threshold: Highest in group
            "THE QUESTION MASTER": question_pct / 0.25,         # Threshold: 25%
            "THE HYPE BEAST (Bonus)": s['hype_count'] / 20.0    # Threshold: arbitrarily 20 hype words
        }

        # Format the output strings for the report
        output_labels = {
            "THE SPAMMER": f"(avg {avg_burst:.1f} msgs in a row)",
            "THE GROUP MOM": f"(caring keyword score: {s['caring_count']})",
            "THE NIGHT OWL": f"({night_pct*100:.1f}% msgs after 11 PM)",
            "THE STORYTELLER": f"(avg {avg_words:.1f} words per msg)",
            "THE DRAMA QUEEN": f"({drama_pct*100:.1f}% ALL-CAPS/exclamation msgs)",
            "THE GHOST": f"(silent {silent_pct*100:.0f}% of days)",
            "THE COMEDIAN": f"({comedian_pct*100:.1f}% comedy words)",
            "THE QUESTION MASTER": f"({question_pct*100:.1f}% questions)",
            "THE HYPE BEAST (Bonus)": f"(used hype words {s['hype_count']} times)"
        }

        # 5. Resolve ties and enforce exclusivity
        # Sort their archetypes by highest score descending
        sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        assigned_archetypes[sender] = {
            'choices': sorted_scores,
            'labels': output_labels
        }

    # Make assignments strictly exclusive (Highest score globally gets it first)
    final_assignments = {}
    taken_archetypes = set()

    # Sort people by who has the most dominating standout trait
    # (i.e., whoever's highest score is the largest across the whole group)
    people_sorted_by_extremity = sorted(
        participants,
        key=lambda p: assigned_archetypes[p]['choices'][0][1],
        reverse=True
    )

    for person in people_sorted_by_extremity:
        for arch_name, score in assigned_archetypes[person]['choices']:
            if arch_name not in taken_archetypes:
                label = assigned_archetypes[person]['labels'][arch_name]
                final_assignments[person] = f"{arch_name} {label}"
                taken_archetypes.add(arch_name)
                break

    # 6. Print the formatted output
    print("PERSONALITY ARCHETYPES")
    max_name_len = max(len(name) for name in participants)

    # Print in original participant order for consistency
    for person in participants:
        print(f"{person:<{max_name_len}} : {final_assignments[person]}")

# To run the feature using the data parsed from Feature 1:
personality_archetypes(messages)

PERSONALITY ARCHETYPES
Rahul : THE SPAMMER (avg 4.5 msgs in a row)
Priya : THE QUESTION MASTER (29.2% questions)
Karan : THE STORYTELLER (avg 55.6 words per msg)
Neha  : THE HYPE BEAST (Bonus) (used hype words 79 times)
Aman  : THE NIGHT OWL (79.8% msgs after 11 PM)
Vikas : THE GHOST (silent 73% of days)


#Feature-8: Final Report

In [ ]:
def generate_final_report(parsed_messages, group_name="Hostel Bois 4ever"):
    """
    Acts as the master execution function. It wraps all the analytical
    components into a single, beautifully formatted terminal report ready
    for screenshotting, complete with headers, dividers, and footers.
    """

    # 1. Compute Top-Level Stats for the Master Banner
    total_messages = len(parsed_messages)

    # Calculate unique members
    participants = set(msg['sender'] for msg in parsed_messages)
    total_members = len(participants)

    # Calculate total days
    first_date_raw = parsed_messages[0]['timestamp'].split(',')[0].strip()
    last_date_raw = parsed_messages[-1]['timestamp'].split(',')[0].strip()
    start_date = datetime.strptime(first_date_raw, "%d/%m/%y")
    end_date = datetime.strptime(last_date_raw, "%d/%m/%y")
    total_days = (end_date - start_date).days + 1

    month_names = {
        "01": "April", "02": "February", "03": "March", "04": "April",
        "05": "May", "06": "June", "07": "July", "08": "August",
        "09": "September", "10": "October", "11": "November", "12": "December"
    }

    first_date_str = f"{str(start_date.day).zfill(2)} {month_names[str(start_date.month).zfill(2)]} 20{str(start_date.year)[-2:]}"
    last_date_str = f"{str(end_date.day).zfill(2)} {month_names[str(end_date.month).zfill(2)]} 20{str(end_date.year)[-2:]}"

    # 2. Print Master Header (Strict formatting per Section 10)
    print("=" * 60)
    print(f"GROUPDNA REPORT : \"{group_name}\"")
    print(f"{total_days} days | {total_messages:,} messages | {total_members} members")
    print("=" * 60)

    print(f"Period: {first_date_str} to {last_date_str}")

    # 3. Call Feature 3 (Busiest Day & Hour)
    most_active_day_and_hour(parsed_messages)
    print() # Empty line for visual hierarchy

    # 4. Call Feature 2's logic (Messages per person)
    # (Assuming group_overview logic is printed here or adapted to not reprint the same banner)
    print("MESSAGES PER PERSON")
    sender_counts = {}
    for msg in parsed_messages:
        sender_counts[msg['sender']] = sender_counts.get(msg['sender'], 0) + 1
    sorted_senders = sorted(sender_counts.items(), key=lambda x: x[1], reverse=True)
    max_name_len = max(len(name) for name in sender_counts.keys())
    for name, count in sorted_senders:
        percentage = (count / total_messages) * 100
        print(f"{name:<{max_name_len}} : {count:>3} ({percentage:>4.1f}%)")
    print()

    # 5. Call Feature 4 (NumPy Heatmap)
    activity_heatmap(parsed_messages)
    print()

    # 6. Call Feature 5 (Top Words Bar Chart)
    top_words(parsed_messages)
    print()

    # 7. Call Feature 6 (Response Times & Silent Streaks)
    response_speed_and_silent_streaks(parsed_messages)
    print()

    # 8. Call Feature 7 (Personality Archetypes)
    personality_archetypes(parsed_messages)

    # 9. Print Master Footer
    print("=" * 60)
    print("Generated by GroupDNA | Built with Python + NumPy")
    print("=" * 60)

# ==========================================
# 1. Run Feature 1 to parse the file
messages = parse_chat('hostel_bois.txt')
#
# 2. Pass the parsed data into the Final Report generator
generate_final_report(messages)

Successfully parsed 3174 messages from 6 participants over 60 days, skipped 4 system messages, 32 media-omitted, 15 deleted messages.
GROUPDNA REPORT : "Hostel Bois 4ever"
60 days | 3,174 messages | 6 members
Period: 01 April 2024 to 30 May 2024
Busiest day : 04 May 2024 (76 messages)
Busiest hour: 18:00 - 19:00 (avg 4 messages per day)

MESSAGES PER PERSON
Rahul : 953 (30.0%)
Priya : 718 (22.6%)
Neha  : 635 (20.0%)
Aman  : 490 (15.4%)
Karan : 354 (11.2%)
Vikas :  24 ( 0.8%)

ACTIVITY HEATMAP (messages by hour)
        00 03 06 09 12 15 18 21
Rahul : ............▒░░▒▒░▓▒░▓▒▒
Priya : .......░▒▓▓▓▓▒▒░░▒▒▓▒░░.
Neha  : .....░..▒▓▓░▒▒░.▒▓▓▓▒░░░
Aman  : ▒▓▒▒▓..................▒
Karan : ........░░▒░▓▒▓▒▒▒▒▓▒░..
Vikas : .......░▓░░.▒▒.░░▓▒▒░░░▒

THIS GROUP'S FAVOURITE WORDS
how      ██████████████████████████████ 321
guys     █████████████████████████████ 318
about    █████████████████████████ 274
today    ████████████████████████ 257
he       ████████████████████ 220
his      ████████████████

## 📝 Project Reflection

**1. What was the hardest part?**

 The hardest part was figuring out the logic for the silent streaks without using pandas, specifically figuring out how to check every single date in the range using the datetime module.

**2. What would you do differently?**

 If I built this again, I would spend more time optimizing the word tokenizer in Feature 5 to handle emojis as separate tokens, which would give a much richer 'top words' output.

**3. My Own Archetype **

 When I ran this code on my own college friend group chat, my assigned archetype was 'THE NIGHT OWL' because it turns out I send 68% of my messages after midnight!